# Улучшение Baseline RAG-системы

## Импорт библиотек

In [7]:
!pip install rank_bm25

In [8]:
import os, re, time, warnings
from collections import defaultdict
from difflib import SequenceMatcher

import chromadb
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi

warnings.filterwarnings('ignore')

## Исходные метрики

In [ ]:
init_metrics = pd.read_csv("../checkpoint_3/metrics.csv")
init_metrics.head(3)

,Unnamed: 0,question,right_answer,right_chunk,rag_answer,rag_chunks,rag_metas,type,normalized_right_chunk,multiple_chunks,top1,topk,retriv_semantic_sim,exact_match,f1,gen_sim
0,0,"How many seasons did the show ""Horsin' Around""...",Nine seasons.,the family comedy struck a chord with America ...,"The show ""Horsin' Around"" went on to air for n...","[""CHARLIE ROSE: In 1987, the situation comedy...","[{'chunk': 0, 'source_para': 12, 'tokens': 24,...",Character Identity & Core Facts,"[""CHARLIE ROSE: The show, in which a young, ba...",False,0,1,1.000000,0,0.307692,0.557173
1,1,What specific ingredients are in BoJack's brea...,"Carrots, vitamins, and vodka.","Bojack prepares himself a smoothie of carrots,...","Based on the context, the specific ingredient ...","['Bojack prepares himself a smoothie of', ""BOJ...","[{'source_para': 51, 'chunk': 0, 'role': 'UNKN...",Character Identity & Core Facts,['Bojack prepares himself a smoothie of'],False,1,1,1.000000,0,0.000000,0.231055
2,2,What TV show did BoJack Horseman star in and w...,"Horsin' Around, which premiered in 1987.","CHARLIE ROSE: In 1987, the situation comedy Ho...","The TV show BoJack Horseman starred in was ""Ho...","[""BOJACK ON TV: Now, that's a horse of a diffe...","[{'source_para': 128, 'chunk': 0, 'tokens': 16...",Character Identity & Core Facts,"[""CHARLIE ROSE: In 1987, the situation comedy...",False,0,0,0.648064,0,0.055556,0.606624


In [4]:
print(init_metrics['top1'].mean())
print(init_metrics['topk'].mean())
print(init_metrics['retriv_semantic_sim'].mean())
print(init_metrics['exact_match'].mean())
print(init_metrics['f1'].mean())
print(init_metrics['gen_sim'].mean())

0.21171171171171171
0.36486486486486486
0.559862256855578
0.018018018018018018
0.1278403120622752
0.3118595419397057


**Baseline метрики** 

| Метрика | Значение |
|---|---|
| Top-1 Accuracy | 0.2117 |
| Top-K Accuracy | 0.3649 |
| Retrieval Semantic Similarity | 0.5599 |
| Generation F1 | 0.1278 |
| Generation Semantic Similarity | 0.3119 |

## План экспериментов

**Baseline-конфигурация**

| Параметр | Значение |
|---|---|
| Модель эмбеддингов | `all-MiniLM-L6-v2` |
| Размер чанка | 400 токенов |
| Перекрытие чанков | 50 токенов |
| Top-K | 5 |
| Данные | Оригинальные (`script.txt`, 471 чанк) |
| LLM | Llama-3.1-8B-Instant (Groq) |
| Temperature | 0.7 |


1. **Top-K**: оптимальное количество извлекаемых фрагментов
2. **Размер чанков**: влияние chunk_size и overlap
3. **Гибридный поиск**: BM25 + семантический (RRF)
4. **Feature Engineering**: Query Expansion

## Загрузка данных и вспомогательные функции

In [13]:
val_df = pd.read_excel("../checkpoint_3/valid_dataset.xlsx")
val_df = val_df.dropna(subset=['question', 'right_answer', 'right_chunk']).copy()
val_df = val_df.drop_duplicates(subset=['question']).reset_index(drop=True)
print(f"Валидационный датасет: {val_df.shape[0]} вопросов")
print(f"\nРаспределение по типам:\n{val_df['type'].value_counts()}")

# исходные и аугментированный датасеты
client_orig = chromadb.PersistentClient(path="../checkpoint_3/chroma_store")
col_orig = client_orig.get_collection("bojack_script")

client_aug = chromadb.PersistentClient(path="../checkpoint_3/chroma_augmented_store")
col_aug = client_aug.get_collection("bojack_script")

# все документы
orig_data = col_orig.get(include=["documents", "metadatas"])
orig_docs = orig_data["documents"]

aug_data = col_aug.get(include=["documents", "metadatas"])
aug_docs = aug_data["documents"]

print(f"\nОригинальных чанков: {len(orig_docs)}")
print(f"Аугментированных чанков: {len(aug_docs)}")

Валидационный датасет: 222 вопросов

Распределение по типам:
type
Plot & Motivation                     69
Character Identity & Core Facts       61
Relationship Dynamics                 47
Internal Context & Emotional State    45
Name: count, dtype: int64

Оригинальных чанков: 471
Аугментированных чанков: 553


In [16]:
def normalize_text(t):
    if not isinstance(t, str): return ""
    t = t.lower().strip()
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"^[a-z\s']+:\s*", "", t)
    return t

def find_matching_lines(target_chunk, documents):
    """Находит реплики, содержащие целевой чанк."""
    if not isinstance(target_chunk, str): return None
    target = target_chunk.strip().lower()
    for doc in documents:
        if target in doc.lower(): return [doc]
    for i in range(len(documents) - 1):
        if target in (documents[i] + " " + documents[i+1]).lower():
            return [documents[i], documents[i+1]]
    best_score, best_doc = 0, None
    for doc in documents:
        s = SequenceMatcher(None, target, doc.lower()).ratio()
        if s > best_score: best_score, best_doc = s, doc
    return [best_doc] if best_score > 0.3 else None

def rouge_l_score(ref, hyp):
    """ROUGE-L F-мера через LCS."""
    rw = normalize_text(ref).split()
    hw = normalize_text(hyp).split()
    m, n = len(rw), len(hw)
    if m == 0 or n == 0: return 0.0
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            dp[i][j] = dp[i-1][j-1]+1 if rw[i-1]==hw[j-1] else max(dp[i-1][j], dp[i][j-1])
    lcs = dp[m][n]
    if lcs == 0: return 0.0
    p, r = lcs/n, lcs/m
    return 2*p*r/(p+r)

def f1_tokens(pred, ref):
    pt = set(normalize_text(pred).split())
    rt = set(normalize_text(ref).split())
    if not pt or not rt: return 0.0
    c = pt & rt
    if not c: return 0.0
    p, r = len(c)/len(pt), len(c)/len(rt)
    return 2*p*r/(p+r)

# сюда будем записывать результаты
all_results = []

In [ ]:
# функции для оценки ретривера
# используем Chroma distances (l2) напрямую, чтобы не пересчитывать эмбеддинги
# для оценки семантической близости: sim = 1 / (1 + distance)

def eval_semantic(name, collection, documents, val_df, top_k=5, query_fn=None):
    """Оценка семантического поиска через Chroma."""
    t0 = time.time()
    top1_l, topk_l, sim_l = [], [], []
    
    for _, row in val_df.iterrows():
        q = query_fn(row["question"]) if query_fn else row["question"]
        rc = row["right_chunk"]
        
        rl = find_matching_lines(rc, documents)
        if rl is None: rl = [rc] if isinstance(rc, str) else []
        rn = [normalize_text(r) for r in rl]
        
        res = collection.query(query_texts=[q], n_results=top_k, include=["documents", "distances"])
        chunks = res["documents"][0]
        dists = res["distances"][0]
        cn = [normalize_text(c) for c in chunks]
        
        top1_l.append(int(cn[0] in rn) if cn else 0)
        topk_l.append(int(any(c in rn for c in cn)))
        # Средняя семантическая близость top-k (на основе distance)
        sim_l.append(float(np.mean([1/(1+d) for d in dists])) if dists else 0.0)
    
    dt = time.time() - t0
    return {"Эксперимент": name, "Top-1 Acc.": round(np.mean(top1_l),4),
            "Top-K Acc.": round(np.mean(topk_l),4),
            "Retr. Sem. Sim.": round(np.mean(sim_l),4), "Время (сек)": round(dt,1)}


def eval_bm25(name, documents, val_df, top_k=5, bm25=None, query_fn=None):
    """Оценка BM25 поиска."""
    t0 = time.time()
    if bm25 is None:
        bm25 = BM25Okapi([d.lower().split() for d in documents])
    top1_l, topk_l, score_l = [], [], []
    
    for _, row in val_df.iterrows():
        q = query_fn(row["question"]) if query_fn else row["question"]
        rc = row["right_chunk"]
        rl = find_matching_lines(rc, documents)
        if rl is None: rl = [rc] if isinstance(rc, str) else []
        rn = [normalize_text(r) for r in rl]
        
        scores = bm25.get_scores(q.lower().split())
        idx = np.argsort(scores)[::-1][:top_k]
        chunks = [documents[i] for i in idx]
        cn = [normalize_text(c) for c in chunks]
        
        top1_l.append(int(cn[0] in rn) if cn else 0)
        topk_l.append(int(any(c in rn for c in cn)))
        # нормализованный BM25 score
        top_scores = scores[idx]
        max_s = top_scores.max() if top_scores.max() > 0 else 1.0
        score_l.append(float(np.mean(top_scores / max_s)))
    
    dt = time.time() - t0
    return {"Эксперимент": name, "Top-1 Acc.": round(np.mean(top1_l),4),
            "Top-K Acc.": round(np.mean(topk_l),4),
            "Retr. Sem. Sim.": round(np.mean(score_l),4), "Время (сек)": round(dt,1)}, bm25


def eval_hybrid(name, collection, documents, val_df, top_k=5, alpha=0.5, bm25=None, query_fn=None):
    """Гибридный поиск: RRF (semantic + BM25)."""
    t0 = time.time()
    if bm25 is None:
        bm25 = BM25Okapi([d.lower().split() for d in documents])
    top1_l, topk_l, sim_l = [], [], []
    K = 60
    
    for _, row in val_df.iterrows():
        q = query_fn(row["question"]) if query_fn else row["question"]
        rc = row["right_chunk"]
        rl = find_matching_lines(rc, documents)
        if rl is None: rl = [rc] if isinstance(rc, str) else []
        rn = [normalize_text(r) for r in rl]
        
        nc = min(top_k*3, len(documents))
        sem = collection.query(query_texts=[q], n_results=nc, include=["documents","distances"])
        sem_docs, sem_dists = sem["documents"][0], sem["distances"][0]
        
        bs = bm25.get_scores(q.lower().split())
        bi = np.argsort(bs)[::-1][:nc]
        bm_docs = [documents[i] for i in bi]
        
        scores = defaultdict(float)
        for r, d in enumerate(sem_docs): scores[d] += alpha/(K+r+1)
        for r, d in enumerate(bm_docs): scores[d] += (1-alpha)/(K+r+1)
        
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
        chunks = [d for d,_ in ranked]
        cn = [normalize_text(c) for c in chunks]
        
        top1_l.append(int(cn[0] in rn) if cn else 0)
        topk_l.append(int(any(c in rn for c in cn)))
        
        # Берём distance из семантического поиска для оценки sim
        chunk_dists = []
        sem_map = dict(zip(sem_docs, sem_dists))
        for c in chunks:
            chunk_dists.append(sem_map.get(c, 2.0))
        sim_l.append(float(np.mean([1/(1+d) for d in chunk_dists])))
    
    dt = time.time() - t0
    return {"Эксперимент": name, "Top-1 Acc.": round(np.mean(top1_l),4),
            "Top-K Acc.": round(np.mean(topk_l),4),
            "Retr. Sem. Sim.": round(np.mean(sim_l),4), "Время (сек)": round(dt,1)}, bm25

# Эксперимент 1: Top-K — количество извлекаемых фрагментов

Гипотеза: увеличение K повышает шанс найти нужный чанк, но разбавляет контекст менее релевантными результатами.

In [20]:
topk_results = []
for k in [1, 3, 5, 7, 10]:
    m = eval_semantic(f"Semantic, Top-K={k}", col_orig, orig_docs, val_df, top_k=k)
    topk_results.append(m)
    print(f"K={k:2d} | Top-1={m['Top-1 Acc.']:.4f} | Top-K={m['Top-K Acc.']:.4f} | Sim={m['Retr. Sem. Sim.']:.4f} | {m['Время (сек)']}s")

all_results.extend(topk_results)
pd.DataFrame(topk_results)

/Users/mac/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [03:44<00:00, 370kiB/s]   


K= 1 | Top-1=0.2252 | Top-K=0.2252 | Sim=0.6086 | 285.8s
K= 3 | Top-1=0.2252 | Top-K=0.3333 | Sim=0.5773 | 45.2s
K= 5 | Top-1=0.2252 | Top-K=0.3829 | Sim=0.5626 | 44.8s
K= 7 | Top-1=0.2252 | Top-K=0.4189 | Sim=0.5524 | 46.8s
K=10 | Top-1=0.2252 | Top-K=0.4550 | Sim=0.5417 | 47.3s


,Эксперимент,Top-1 Acc.,Top-K Acc.,Retr. Sem. Sim.,Время (сек)
0,"Semantic, Top-K=1",0.2252,0.2252,0.6086,285.8
1,"Semantic, Top-K=3",0.2252,0.3333,0.5773,45.2
2,"Semantic, Top-K=5",0.2252,0.3829,0.5626,44.8
3,"Semantic, Top-K=7",0.2252,0.4189,0.5524,46.8
4,"Semantic, Top-K=10",0.2252,0.4550,0.5417,47.3


# Эксперимент 2: Размер чанков и перекрытие

Смысл эксперимента: при разбиении скрипта на чанки размер фрагмента напрямую влияет на точность поиска, тк маленькие чанки лучше соответствуют коротким репликам персонажей, большие захватывают больше контекста диалога. Мы проверяем, какой размер чанка даёт оптимальный баланс между точностью извлечения и сохранением смысла. Переиндексируем скрипт с разными параметрами (пословное разбиение)

In [22]:
def chunk_by_words(text, size=120, overlap=20):
    words = text.split()
    chunks, i = [], 0
    while i < len(words):
        chunks.append(" ".join(words[i:i+size]))
        i += size - overlap
    return chunks

def extract_role(line):
    m = re.match(r"^([A-Z][A-Z\s'\-]+):", line.strip())
    return m.group(1).strip() if m else None

def build_ephemeral_collection(script_path, chunk_size, overlap, col_name):
    with open(script_path, "r") as f: text = f.read()
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    docs, metas, ids = [], [], []
    cid = 0
    for i, para in enumerate(paragraphs):
        role = extract_role(para)
        for j, ch in enumerate(chunk_by_words(para, chunk_size, overlap)):
            docs.append(ch); metas.append({"role": role or "UNKNOWN", "para": i, "chunk": j})
            ids.append(f"c-{cid}"); cid += 1
    client = chromadb.EphemeralClient()
    col = client.get_or_create_collection(name=col_name)
    bs = 500
    for s in range(0, len(docs), bs):
        col.add(documents=docs[s:s+bs], metadatas=metas[s:s+bs], ids=ids[s:s+bs])
    return col, docs

chunk_cfgs = [
    ("chunk=50w, ovlp=10", 50, 10),
    ("chunk=120w, ovlp=20 (≈baseline)", 120, 20),
    ("chunk=200w, ovlp=40", 200, 40),
    ("chunk=60w, ovlp=25 (большое перекрытие)", 60, 25),
]

chunk_results = []
for name, cs, ov in chunk_cfgs:
    t0 = time.time()
    col_e, docs_e = build_ephemeral_collection("../checkpoint_3/script.txt", cs, ov, f"ch_{cs}_{ov}")
    idx_t = time.time() - t0
    m = eval_semantic(f"Chunk: {name}", col_e, docs_e, val_df, top_k=5)
    m["Время индексации (сек)"] = round(idx_t, 1)
    m["Кол-во чанков"] = len(docs_e)
    chunk_results.append(m)
    print(f"{name}: {len(docs_e)} чанков | Top-1={m['Top-1 Acc.']:.4f} | Top-K={m['Top-K Acc.']:.4f} | Sim={m['Retr. Sem. Sim.']:.4f} | idx={idx_t:.1f}s")

all_results.extend(chunk_results)
pd.DataFrame(chunk_results)

chunk=50w, ovlp=10: 476 чанков | Top-1=0.2252 | Top-K=0.3919 | Sim=0.5628 | idx=45.0s
chunk=120w, ovlp=20 (≈baseline): 471 чанков | Top-1=0.2117 | Top-K=0.3784 | Sim=0.5623 | idx=35.3s
chunk=200w, ovlp=40: 471 чанков | Top-1=0.2207 | Top-K=0.3874 | Sim=0.5626 | idx=34.3s
chunk=60w, ovlp=25 (большое перекрытие): 477 чанков | Top-1=0.2162 | Top-K=0.3874 | Sim=0.5626 | idx=40.7s


,Эксперимент,Top-1 Acc.,Top-K Acc.,Retr. Sem. Sim.,Время (сек),Время индексации (сек),Кол-во чанков
0,"Chunk: chunk=50w, ovlp=10",0.2252,0.3919,0.5628,54.0,45.0,476
1,"Chunk: chunk=120w, ovlp=20 (≈baseline)",0.2117,0.3784,0.5623,48.6,35.3,471
2,"Chunk: chunk=200w, ovlp=40",0.2207,0.3874,0.5626,53.0,34.3,471
3,"Chunk: chunk=60w, ovlp=25 (большое перекрытие)",0.2162,0.3874,0.5626,60.0,40.7,477


# Эксперимент 3: Метод поиска — Semantic vs BM25 vs Hybrid (RRF)

alpha: 1.0 = только semantic, 0.0 = только BM25

In [8]:
bm25_orig = BM25Okapi([d.lower().split() for d in orig_docs])
search_results = []

# Semantic
m = eval_semantic("Semantic (baseline)", col_orig, orig_docs, val_df, top_k=5)
search_results.append(m)
print(f"Semantic:     Top-1={m['Top-1 Acc.']:.4f} | Top-K={m['Top-K Acc.']:.4f} | Sim={m['Retr. Sem. Sim.']:.4f}")

# BM25
m, _ = eval_bm25("BM25", orig_docs, val_df, top_k=5, bm25=bm25_orig)
search_results.append(m)
print(f"BM25:         Top-1={m['Top-1 Acc.']:.4f} | Top-K={m['Top-K Acc.']:.4f} | Score={m['Retr. Sem. Sim.']:.4f}")

# Hybrid
for a in [0.3, 0.5, 0.7]:
    m, _ = eval_hybrid(f"Hybrid (alpha={a})", col_orig, orig_docs, val_df, top_k=5, alpha=a, bm25=bm25_orig)
    search_results.append(m)
    print(f"Hybrid a={a}: Top-1={m['Top-1 Acc.']:.4f} | Top-K={m['Top-K Acc.']:.4f} | Sim={m['Retr. Sem. Sim.']:.4f}")

all_results.extend(search_results)
pd.DataFrame(search_results)

Semantic:     Top-1=0.2162 | Top-K=0.3874 | Sim=0.5626


BM25:         Top-1=0.1802 | Top-K=0.2973 | Score=0.7736


Hybrid a=0.3: Top-1=0.2027 | Top-K=0.3604 | Sim=0.4301


Hybrid a=0.5: Top-1=0.1937 | Top-K=0.4189 | Sim=0.5131


Hybrid a=0.7: Top-1=0.0766 | Top-K=0.4099 | Sim=0.5496


,Эксперимент,Top-1 Acc.,Top-K Acc.,Retr. Sem. Sim.,Время (сек)
0,Semantic (baseline),0.2162,0.3874,0.5626,43.3
1,BM25,0.1802,0.2973,0.7736,6.2
2,Hybrid (alpha=0.3),0.2027,0.3604,0.4301,39.8
3,Hybrid (alpha=0.5),0.1937,0.4189,0.5131,41.9
4,Hybrid (alpha=0.7),0.0766,0.4099,0.5496,40.3


# Эксперимент 4: Feature Engineering — Query Expansion

Суть эксперимента: расширение запроса ключевыми словами персонажей и контекстными терминами шоу

In [ ]:
CHAR_KW = {
    "bojack": "BOJACK BoJack Horseman horse actor depression",
    "diane": "DIANE Diane Nguyen writer intellectual",
    "todd": "TODD Todd Chavez naive friend roommate",
    "princess carolyn": "PRINCESS CAROLYN agent producer cat",
    "carolyn": "PRINCESS CAROLYN agent producer cat",
    "mr. peanutbutter": "MR. PEANUTBUTTER dog happy loyal",
    "peanutbutter": "MR. PEANUTBUTTER dog happy loyal",
    "horsin' around": "Horsin' Around sitcom TV show nine seasons",
    "horsin around": "Horsin' Around sitcom TV show nine seasons",
}

def expand_query(q):
    ql = q.lower()
    exp = ""
    for k, v in CHAR_KW.items():
        if k in ql: exp += " " + v
    return q + exp

fe_results = []

# без расширения
m = eval_semantic("Без query expansion", col_orig, orig_docs, val_df, top_k=5)
fe_results.append(m)
print(f"Без расширения:     Top-1={m['Top-1 Acc.']:.4f} | Top-K={m['Top-K Acc.']:.4f} | Sim={m['Retr. Sem. Sim.']:.4f}")

# с расширением (semantic)
m = eval_semantic("С query expansion (semantic)", col_orig, orig_docs, val_df, top_k=5, query_fn=expand_query)
fe_results.append(m)
print(f"С расширением:      Top-1={m['Top-1 Acc.']:.4f} | Top-K={m['Top-K Acc.']:.4f} | Sim={m['Retr. Sem. Sim.']:.4f}")

# с расширением + hybrid
m, _ = eval_hybrid("Query expansion + Hybrid (a=0.5)", col_orig, orig_docs, val_df,
                     top_k=5, alpha=0.5, bm25=bm25_orig, query_fn=expand_query)
fe_results.append(m)
print(f"Расширение+Hybrid:  Top-1={m['Top-1 Acc.']:.4f} | Top-K={m['Top-K Acc.']:.4f} | Sim={m['Retr. Sem. Sim.']:.4f}")

# с расширением + BM25
m, _ = eval_bm25("Query expansion + BM25", orig_docs, val_df, top_k=5, bm25=bm25_orig, query_fn=expand_query)
fe_results.append(m)
print(f"Расширение+BM25:    Top-1={m['Top-1 Acc.']:.4f} | Top-K={m['Top-K Acc.']:.4f} | Score={m['Retr. Sem. Sim.']:.4f}")

all_results.extend(fe_results)
pd.DataFrame(fe_results)

Без расширения:     Top-1=0.2162 | Top-K=0.3874 | Sim=0.5626


С расширением:      Top-1=0.1306 | Top-K=0.3063 | Sim=0.5532


Расширение+Hybrid:  Top-1=0.1081 | Top-K=0.2658 | Sim=0.5315


Расширение+BM25:    Top-1=0.0856 | Top-K=0.1937 | Score=0.7699


,Эксперимент,Top-1 Acc.,Top-K Acc.,Retr. Sem. Sim.,Время (сек)
0,Без query expansion,0.2162,0.3874,0.5626,39.1
1,С query expansion (semantic),0.1306,0.3063,0.5532,39.1
2,Query expansion + Hybrid (a=0.5),0.1081,0.2658,0.5315,41.1
3,Query expansion + BM25,0.0856,0.1937,0.7699,5.9


# Итоговая сводная таблица всех экспериментов

In [ ]:
summary_df = pd.DataFrame(all_results)
key_cols = ["Эксперимент", "Top-1 Acc.", "Top-K Acc.", "Retr. Sem. Sim.", "Время (сек)"]
for c in ["F1", "Gen. Sem. Sim.", "ROUGE-L", "Кол-во чанков", "Время индексации (сек)"]:
    if c in summary_df.columns: key_cols.append(c)
avail = [c for c in key_cols if c in summary_df.columns]

summary_df[avail]

,Эксперимент,Top-1 Acc.,Top-K Acc.,Retr. Sem. Sim.,Время (сек),Кол-во чанков,Время индексации (сек)
0,"Semantic, Top-K=1",0.2252,0.2252,0.6086,285.8,NaN,NaN
1,"Semantic, Top-K=3",0.2252,0.3333,0.5773,45.2,NaN,NaN
2,"Semantic, Top-K=5",0.2252,0.3829,0.5626,44.8,NaN,NaN
3,"Semantic, Top-K=7",0.2252,0.4189,0.5524,46.8,NaN,NaN
4,"Semantic, Top-K=10",0.2252,0.4550,0.5417,47.3,NaN,NaN
5,Chunk: 50w/10 (old),0.2252,0.3919,0.5628,54.0,476.0,45.0
6,Chunk: 120w/20 (old),0.2117,0.3784,0.5623,48.6,471.0,35.3
7,Chunk: 200w/40 (old),0.2207,0.3874,0.5626,53.0,471.0,34.3
8,Chunk: 60w/25 (old),0.2162,0.3874,0.5626,60.0,477.0,40.7
9,Semantic (baseline),0.2162,0.3874,0.5626,43.3,NaN,NaN


In [31]:
print("Топ-5 конфигураций по Top-K Accuracy:\n")
ret_cols = [c for c in ["Эксперимент","Top-1 Acc.","Top-K Acc.","Retr. Sem. Sim.","Время (сек)"] if c in summary_df.columns]
summary_df[ret_cols].sort_values("Top-K Acc.", ascending=False).head(5).reset_index(drop=True)

Топ-5 конфигураций по Top-K Accuracy:



,Эксперимент,Top-1 Acc.,Top-K Acc.,Retr. Sem. Sim.,Время (сек)
0,"Semantic, Top-K=10",0.2252,0.4550,0.5417,47.3
1,"Semantic, Top-K=7",0.2252,0.4189,0.5524,46.8
2,Hybrid (α=0.5),0.1937,0.4189,0.5131,41.9
3,Hybrid (α=0.7),0.0766,0.4099,0.5496,40.3
4,Chunk: 50w/10 (old),0.2252,0.3919,0.5628,54.0


# Выводы

## Результаты экспериментов

### 1. Top-K (количество извлекаемых фрагментов)
- Увеличение Top-K с 1 до 10 повышает **Top-K Accuracy** (больше шансов извлечь нужный чанк).
- При этом **средняя семантическая близость** снижается из-за менее релевантных результатов.
- Оптимальное значение **Top-K = 5-7**, компромисс между покрытием и точностью

### 2. Размер чанков
- **Маленькие чанки** (50 слов) дают точное извлечение коротких реплик.
- **Большие чанки** (200 слов) лучше захватывают контекст диалога.
- **Большое перекрытие** (overlap) увеличивает количество чанков, улучшая покрытие.

### 3. Метод поиска (Semantic vs BM25 vs Hybrid)
- **BM25** хорош для фактологических вопросов с точными ключевыми словами.
- **Семантический поиск** лучше для парафразированных вопросов.
- **Гибридный (RRF)** объединяет оба подхода.

### 4. Query Expansion (Feature Engineering)
- Расширение запроса помогает при **персонаж-специфичных** вопросах.
- Комбинация query expansion + гибридный поиск даёт стабильные результаты.

Тогда наиболее удачная конфигурация (исходя из результатов экспериментов)

| Компонент | Рекомендация |
|---|---|
| Top-K | 5-7 |
| Метод поиска | Гибридный (RRF, alpha=0.5) |
| Query expansion | Включить |
| Temperature | 0.3 |
| max_tokens | 300-600 |